<div style="background:linear-gradient(135deg,#51247a 0%,#7a3fa0 100%);padding:22px 26px;border-radius:10px;margin-bottom:1.2rem;border-bottom:4px solid #f0a500;">
<div style="font-size:2.2rem;margin-bottom:6px;">💬</div>
<div style="color:white;font-size:1.5rem;font-weight:700;">Sentiment Explorer</div>
<div style="color:rgba(255,255,255,.8);font-size:.92rem;margin-top:4px;">Score texts for positive/negative polarity and eight basic emotions<br><a href="https://ladal.edu.au/sentiment.html" style="color:#f0c060;font-size:.82rem;">&#x2192; See the full tutorial</a></div>
</div>


**What this tool does:** Analyses the emotional tone of your texts using the [NRC Word-Emotion Association Lexicon](https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm), measuring anger, fear, anticipation, trust, surprise, sadness, joy, and disgust.

**Steps:** Upload files → Run Analysis → Download results


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 1 &mdash; Set up the environment</b></div>


In [ ]:
suppressPackageStartupMessages(library(IRdisplay))
IRdisplay::display_html('
<style>
  .jp-InputArea-editor { display: none !important; }
  .jp-CodeCell .jp-Cell-inputWrapper { display: none !important; }
  .show-code .jp-CodeCell .jp-Cell-inputWrapper { display: flex !important; }
  .ladal-toolbar {
    position: sticky; top: 0; z-index: 999;
    background: #51247a; color: white;
    padding: 8px 18px; border-radius: 0 0 8px 8px;
    display: flex; align-items: center; gap: 14px;
    font-family: sans-serif; font-size: .85rem;
    box-shadow: 0 2px 8px rgba(81,36,122,.25);
  }
  .ladal-toolbar b { font-size: 1rem; }
  .ladal-toolbar button {
    background: rgba(255,255,255,.18); border: 1px solid rgba(255,255,255,.4);
    color: white; padding: 4px 12px; border-radius: 4px;
    cursor: pointer; font-size: .8rem; font-weight: 600;
  }
  .ladal-toolbar button:hover { background: rgba(255,255,255,.32); }
</style>
<div class="ladal-toolbar">
  <b>&#x1F4D3; LADAL Interactive Notebook</b>
  <button onclick="document.body.classList.toggle('show-code');
    this.textContent = document.body.classList.contains('show-code')
      ? 'Hide Code' : 'Show Code'">Show Code</button>
  <span style="opacity:.7;font-size:.78rem;">
    Code is hidden by default &mdash; click to toggle
  </span>
</div>
')


In [ ]:
# ── Suppress startup messages & load display helpers ────────────────
suppressPackageStartupMessages(library(IRdisplay))
suppressPackageStartupMessages(library(IRkernel))

# ── Colour-coded display helpers ────────────────────────────────────
.box <- function(msg, bg, border, icon="") {
  IRdisplay::display_html(paste0(
    '<div style="background:', bg, ';border-left:4px solid ', border,
    ';border-radius:6px;padding:11px 16px;margin:.6rem 0;',
    'font-family:sans-serif;font-size:.9rem;">',
    if (nzchar(icon)) paste0('<b>', icon, '</b> ') else '',
    msg, '</div>'))
}
ok   <- function(msg) .box(msg, "#eafaf1", "#27ae60", "&#x2705;")
warn <- function(msg) .box(msg, "#fff8e1", "#f0a500", "&#x26A0;&#xFE0F;")
err  <- function(msg) .box(msg, "#fff0f0", "#e74c3c", "&#x274C;")
info <- function(msg) .box(msg, "#f4f0f8", "#51247a", "&#x2139;&#xFE0F;")

# ── Progress bar helper ──────────────────────────────────────────────
.prog <- function(label, value, max=100) {
  pct <- round(100 * value / max)
  IRdisplay::display_html(paste0(
    '<div style="font-family:sans-serif;font-size:.85rem;margin:.4rem 0;">',
    '<span style="color:#51247a;font-weight:600;">', label, '</span><br>',
    '<div style="background:#e8e4f0;border-radius:4px;height:10px;margin-top:4px;">',
    '<div style="background:#51247a;width:', pct, '%;height:10px;',
    'border-radius:4px;transition:width .3s;"></div></div>',
    '<span style="color:#888;font-size:.78rem;">', pct, '%</span></div>'))
}

# ── Upload instructions ──────────────────────────────────────────────
upload_instructions <- function(folder="MyTexts", filetype="txt") {
  IRdisplay::display_html(paste0(
    '<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;',
    'padding:18px 22px;margin:.8rem 0;font-family:sans-serif;">',
    '<b style="color:#51247a;font-size:1rem;">&#x1F4C2; Upload your files</b><br><br>',
    '<ol style="margin:0;padding-left:1.4em;font-size:.9rem;line-height:1.8;">',
    '<li>Find the <b>file browser panel</b> on the left side of the screen.</li>',
    '<li>Double-click the <b><code>', folder, '</code></b> folder to open it.</li>',
    '<li><b>Drag and drop</b> your <code>.', filetype, '</code> files into that folder.</li>',
    '<li>Come back here and click <b>Run Analysis</b> below.</li>',
    '</ol>',
    '<p style="margin:.6rem 0 0 0;font-size:.82rem;color:#888;">',
    'Only <code>.', filetype, '</code> files are accepted. ',
    'You can upload multiple files at once.</p></div>'))
}

# ── Load plain-text files ────────────────────────────────────────────
load_texts <- function(folder = "notebooks/MyTexts") {
  files <- list.files(folder, pattern = "\.txt$",
                      full.names = TRUE, recursive = FALSE)
  if (length(files) == 0)
    stop("No .txt files found in '", folder, "'. ",
         "Please upload files into the ", basename(folder), " folder.")
  texts <- vapply(files, function(f)
    paste(readLines(f, warn = FALSE, encoding = "UTF-8"), collapse = " "),
    character(1))
  names(texts) <- tools::file_path_sans_ext(basename(files))
  texts
}

# ── Save texts ───────────────────────────────────────────────────────
save_texts <- function(texts, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  for (nm in names(texts))
    writeLines(texts[[nm]], file.path(folder, paste0(nm, ".txt")))
}

# ── Save Excel ───────────────────────────────────────────────────────
save_excel <- function(df, filename, folder = "notebooks/MyOutput") {
  dir.create(folder, showWarnings = FALSE, recursive = TRUE)
  writexl::write_xlsx(as.data.frame(df), file.path(folder, filename))
}

ok("Environment ready. Scroll down to configure and run your analysis.")


<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 2 📂 &mdash; Upload your texts</b></div>


In [ ]:
upload_instructions("MyTexts", "txt")

<div style="background:#51247a;color:white;padding:8px 16px;border-radius:6px;margin:1.2rem 0 .5rem 0;font-family:sans-serif;"><b>Step 3 &mdash; Run analysis</b></div>


In [ ]:
suppressPackageStartupMessages(library(ipywidgets))

w_btn <- ipywidgets::Button(description="  Run Analysis",
           button_style="primary", icon="heart")
w_out <- ipywidgets::Output()
ipywidgets::display(ipywidgets::VBox(list(w_btn, w_out)))

ipywidgets::observe(w_btn, "clicks", function(change) {
  ipywidgets::with_output(w_out, {
    tryCatch({
      suppressPackageStartupMessages({
        library(syuzhet); library(dplyr); library(tidyr)
        library(ggplot2); library(writexl)
      })
      .prog("Loading texts...", 10)
      texts <- load_texts("notebooks/MyTexts")
      ok(paste("Loaded", length(texts), "file(s)"))

      .prog("Scoring sentiment (NRC lexicon)...", 30)
      senti_list <- lapply(seq_along(texts), function(i) {
        nm    <- names(texts)[i]
        .prog(paste0("Analysing: ", nm, " (", i, "/", length(texts), ")"),
              30 + 40*(i/length(texts)))
        words  <- tolower(unlist(strsplit(texts[[nm]], "[^a-z]+")))
        words  <- words[nzchar(words)]
        if (length(words)==0) return(NULL)
        scores <- syuzhet::get_nrc_sentiment(words)
        scores$word <- words; scores$document <- nm
        scores
      })
      senti <- dplyr::bind_rows(senti_list)

      .prog("Building summary...", 78)
      emo_cols <- c("anger","anticipation","disgust","fear",
                    "joy","sadness","surprise","trust")
      senti_sum <- senti |>
        dplyr::group_by(document) |>
        dplyr::summarise(words=n(), dplyr::across(all_of(c(emo_cols,"negative","positive")), sum),
                         sentiment=positive-negative, .groups="drop")
      print(senti_sum)

      .prog("Plotting emotion profiles...", 88)
      emo_pal <- c(anger="#e74c3c", anticipation="#f39c12", disgust="#8e44ad",
                   fear="#2c3e50", joy="#27ae60", sadness="#3498db",
                   surprise="#1abc9c", trust="#d35400")
      p <- senti_sum |>
        tidyr::pivot_longer(all_of(emo_cols), names_to="emotion", values_to="count") |>
        ggplot(aes(x=emotion, y=count, fill=emotion)) +
        geom_col(width=.7, show.legend=FALSE) +
        facet_wrap(~document, scales="free_y") +
        scale_fill_manual(values=emo_pal) +
        coord_flip() + theme_bw(base_size=11) +
        labs(title="Emotion profile by document", x=NULL, y="Word count")
      print(p)

      .prog("Saving results...", 95)
      save_excel(senti,     "sentiment_words.xlsx")
      save_excel(senti_sum, "sentiment_summary.xlsx")
      ggsave("notebooks/MyOutput/sentiment_plot.png", bg="white", width=9, height=5, dpi=200)
      .prog("Done.", 100)
      ok("Saved <b>sentiment_words.xlsx</b>, <b>sentiment_summary.xlsx</b>, and <b>sentiment_plot.png</b> to MyOutput.")
    }, error=function(e) err(conditionMessage(e)))
  })
})


---

**Please also cite the NRC lexicon:** Mohammad & Turney (2013). Crowdsourcing a Word-Emotion Association Lexicon. *Computational Intelligence* 29(3), 436–465.


---

### Citation

Schweinberger, Martin. (2025). *LADAL Sentiment Explorer*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025sentiment,
  author = {Schweinberger, Martin},
  title  = {LADAL Sentiment Explorer},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
sessionInfo()